In [1]:
import pandas as pd
import numpy as np
import requests
import io
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.impute import SimpleImputer

In [2]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+ps&format=csv'
print('Fetching exoplanet data from NASA archive...')
response = requests.get(url, timeout=180)
df_raw = pd.read_csv(io.StringIO(response.text), comment='#', low_memory=False)
print(f'Dataset shape: {df_raw.shape}')
df_raw.head(3)

Fetching exoplanet data from NASA archive...
Dataset shape: (39537, 355)


,pl_name,pl_letter,hostname,hd_name,hip_name,tic_id,gaia_dr2_id,gaia_dr3_id,default_flag,pl_refname,...,sy_jmagerr1,sy_jmagerr2,sy_jmagstr,sy_hmag,sy_hmagerr1,sy_hmagerr2,sy_hmagstr,sy_kmag,sy_kmagerr1,sy_kmagerr2
0,HD 156279 b,b,HD 156279,HD 156279,HIP 84171,TIC 232606278,Gaia DR2 1631084478574318976,Gaia DR3 1631084478574318976,0,<a refstr=DIAZ_ET_AL__2012 href=https://ui.ads...,...,0.018,-0.018,6.677&plusmn;0.018,6.346,0.026,-0.026,6.346&plusmn;0.026,6.269,0.023,-0.023
1,GJ 832 b,b,GJ 832,HD 204961,HIP 106440,TIC 139754153,Gaia DR2 6562924609150908416,Gaia DR3 6562924609150908416,0,<a refstr=BONFILS_ET_AL__2013 href=https://ui....,...,0.032,-0.032,5.349&plusmn;0.032,4.766,0.256,-0.256,4.766&plusmn;0.256,4.501,0.018,-0.018
2,GJ 1214 b,b,GJ 1214,NaN,NaN,TIC 467929202,Gaia DR2 4393265392167891712,Gaia DR3 4393265392168829056,0,<a refstr=CARTER_ET_AL__2011 href=https://ui.a...,...,0.024,-0.024,9.750&plusmn;0.024,9.094,0.024,-0.024,9.094&plusmn;0.024,8.782,0.020,-0.020


In [3]:
print('Top 20 columns by missing %:')
print((df_raw.isnull().mean() * 100).sort_values(ascending=False).head(20))

Top 20 columns by missing %:
sy_kepmagerr1       100.000000
sy_kepmagerr2       100.000000
sy_icmagerr1         99.898829
sy_icmagerr2         99.898829
pl_occdeperr1        99.886183
sy_icmag             99.886183
pl_occdeperr2        99.886183
sy_icmagstr          99.886183
pl_occdepstr         99.878595
pl_occdep            99.878595
pl_occdeplim         99.878595
pl_trueobliqerr1     99.838126
pl_trueobliqerr2     99.838126
pl_trueobliqstr      99.820421
pl_trueobliqlim      99.820421
pl_trueobliq         99.820421
pl_cmasseerr1        99.787541
pl_cmassjerr2        99.787541
pl_cmassjerr1        99.787541
pl_cmasseerr2        99.787541
dtype: float64


In [4]:
FEATURE_COLS = ['pl_orbper','pl_rade','pl_masse','pl_orbsmax','pl_orbeccen','pl_orbincl','st_teff','st_rad','st_mass','st_logg','st_met','sy_dist','sy_vmag']
available_features = [c for c in FEATURE_COLS if c in df_raw.columns]
print('Available features:', available_features)
TARGET_COL = 'pl_controv_flag'
if TARGET_COL not in df_raw.columns:
    df_raw['pl_controv_flag'] = (df_raw['pl_masse'].fillna(0) > 13).astype(int)
df = df_raw[available_features + [TARGET_COL]].copy()
df[TARGET_COL] = df[TARGET_COL].fillna(0).astype(int)
df_clean = df.dropna(subset=available_features, thresh=int(len(available_features) * 0.5))
df_clean = df_clean.drop_duplicates(subset=available_features)
print(f'Clean dataset: {df_clean.shape}')

Available features: ['pl_orbper', 'pl_rade', 'pl_masse', 'pl_orbsmax', 'pl_orbeccen', 'pl_orbincl', 'st_teff', 'st_rad', 'st_mass', 'st_logg', 'st_met', 'sy_dist', 'sy_vmag']
Clean dataset: (29586, 14)


In [5]:
n_cols = 3
n_rows = (len(available_features) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes_flat = axes.flatten()
for i, col in enumerate(available_features):
    data = df_clean[col].dropna()
    axes_flat[i].hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes_flat[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.2g}')
    axes_flat[i].set_title(col, fontsize=10, fontweight='bold')
    axes_flat[i].legend(fontsize=7)
for j in range(len(available_features), len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

Saved: feature_distributions.png


In [6]:
corr_data = df_clean[available_features].corr()
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_data, dtype=bool))
sns.heatmap(corr_data, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: correlation_heatmap.png')

Saved: correlation_heatmap.png


In [7]:
target_counts = df_clean[TARGET_COL].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(target_counts.index.astype(str), target_counts.values, color=['green','red'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Controversial Flag')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
if 'pl_orbper' in df_clean.columns and 'pl_rade' in df_clean.columns:
    scatter_df = df_clean[['pl_orbper','pl_rade',TARGET_COL]].dropna()
    scatter_df = scatter_df[(scatter_df['pl_orbper'] > 0) & (scatter_df['pl_rade'] > 0)]
    colors = scatter_df[TARGET_COL].map({0: 'green', 1: 'red'})
    axes[1].scatter(np.log10(scatter_df['pl_orbper']), scatter_df['pl_rade'], c=colors, alpha=0.5, s=15)
    axes[1].set_xlabel('log10(Orbital Period [days])')
    axes[1].set_ylabel('Planet Radius [Earth Radii]')
    axes[1].set_title('Orbital Period vs Planet Radius', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('target_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: target_analysis.png')

Saved: target_analysis.png


In [8]:
X = df_clean[available_features].copy()
y = df_clean[TARGET_COL].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_train_imp = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_imp = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imp)
model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
print(f'Accuracy: {acc:.4f}')
print(f'ROC-AUC:  {roc_auc:.4f}')
print(classification_report(y_test, y_pred))

Train: (23668, 13), Test: (5918, 13)
Accuracy: 0.9975
ROC-AUC:  0.9883
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5895
           1       1.00      0.35      0.52        23

    accuracy                           1.00      5918
   macro avg       1.00      0.67      0.76      5918
weighted avg       1.00      1.00      1.00      5918



In [9]:
importances = model.feature_importances_
feat_imp_df = pd.DataFrame({'feature': available_features, 'importance': importances}).sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
bar_colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feat_imp_df)))
bars = ax.barh(feat_imp_df['feature'], feat_imp_df['importance'], color=bar_colors)
ax.set_xlabel('Importance Score', fontsize=11)
ax.set_title('Feature Importance - Random Forest', fontsize=13, fontweight='bold')
for bar, val in zip(bars, feat_imp_df['importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

Saved: feature_importance.png


In [10]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Confirmed','Controversial'], yticklabels=['Confirmed','Controversial'], linewidths=0.5)
axes[0].set_title('Confusion Matrix - Random Forest', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)
axes[1].plot(fpr, tpr, color='blue', linewidth=2.5, label=f'ROC-AUC = {roc_auc:.3f}')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve - Random Forest', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('model_performance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: model_performance.png')

Saved: model_performance.png


In [11]:
feature_stats = {}
for col in available_features:
    col_data = df_clean[col].dropna()
    feature_stats[col] = {'mean': float(col_data.mean()), 'median': float(col_data.median()), 'std': float(col_data.std()), 'min': float(col_data.min()), 'max': float(col_data.max()), 'q25': float(col_data.quantile(0.25)), 'q75': float(col_data.quantile(0.75))}
model_bundle = {
    'model': model, 'model_name': 'Random Forest', 'imputer': imputer, 'scaler': scaler,
    'feature_cols': available_features, 'target_col': TARGET_COL, 'feature_stats': feature_stats,
    'accuracy': acc, 'roc_auc': roc_auc, 'report': classification_report(y_test, y_pred, output_dict=True),
    'confusion_matrix': cm.tolist(),
    'roc_curve': {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'thresholds': thresholds.tolist()},
    'feature_importance': feat_imp_df.set_index('feature')['importance'].to_dict(),
    'target_distribution': target_counts.to_dict(),
    'correlation_matrix': corr_data.to_dict(),
    'df_sample': df_clean[available_features + [TARGET_COL]].head(2000).to_dict(orient='records')
}
with open('model.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
size_mb = os.path.getsize('model.pkl') / 1024 / 1024
print(f'model.pkl saved! Size: {size_mb:.2f} MB')
print(f'Accuracy: {acc:.4f} | ROC-AUC: {roc_auc:.4f}')
print(f'Features: {available_features}')

model.pkl saved! Size: 3.12 MB
Accuracy: 0.9975 | ROC-AUC: 0.9883
Features: ['pl_orbper', 'pl_rade', 'pl_masse', 'pl_orbsmax', 'pl_orbeccen', 'pl_orbincl', 'st_teff', 'st_rad', 'st_mass', 'st_logg', 'st_met', 'sy_dist', 'sy_vmag']
